# 10 - Evolução do baseline (LightGBM): preços quebrados (P3)

Este notebook mede se o alerta do baseline evolui quando saímos dos erros extremos dos rótulos e passamos a testar erros moderados, e corrige a MAD, que no baseline foi calculada em linhas que o modelo já tinha visto.

**Entradas:** tabelas `squad1/ml/features_precos_observacoes` (292.000 linhas) e `squad1/ml/features_precos_sku` (3.805 linhas), geradas no notebook 08. O modelo base é o mesmo do notebook 09 (7 colunas, 68 árvores).

**Saídas:** recall do alerta em erros sintéticos de 2 a 8 vezes, taxa de falso alarme nas linhas normais e a comparação entre a versão 1 (baseline) e a versão 2 (MAD fora do treino).

**Como rodar:** use "Executar tudo", sempre na ordem. A célula 2 instala o lightgbm (%pip) e precisa ficar logo no começo, antes de qualquer outra.

In [0]:
%pip install lightgbm python-dotenv

In [0]:
# ------------------------------------------------------------
# Configuração
# Importações, acesso ao lake (ADLS) e a função de leitura de tabelas Delta.
# Esta célula não lê nem grava dados.
# ------------------------------------------------------------
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

# Lê o arquivo .env (mesma pasta do notebook), onde ficam as credenciais do ADLS.
load_dotenv(".env")

# Endereço do lake: conta de armazenamento e pasta raiz da Squad 1.
STORAGE_ACCOUNT = "internshipdatalake"
lakehouse_base = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Credenciais de acesso ao ADLS (autenticação OAuth com um aplicativo do Azure).
# No Serverless o spark.conf.set de credenciais é bloqueado,
# por isso elas vão em cada leitura, pelo .options(**adls_options).
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

def ler_delta_spark(caminho):
    """Lê uma tabela Delta do ADLS.

    caminho: endereço abfss:// da tabela (ex.: .../ml/features_precos_sku).
    Retorna um DataFrame do Spark.
    """
    return spark.read.format("delta").options(**adls_options).load(caminho)

In [0]:
# ------------------------------------------------------------
# Carga dos dados e separação treino / teste
# Lê as duas tabelas de features geradas no notebook 08, traz as observações
# para o pandas e separa treino (folds 1 a 4) e teste (fold 0), como no notebook 09.
# Também define as colunas do modelo base e a função de erro (MAE).
# Não altera nada no lake: só lê.
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import lightgbm as lgb

obs_path = f"{lakehouse_base}/ml/features_precos_observacoes"  # uma linha por observação
sku_path = f"{lakehouse_base}/ml/features_precos_sku"          # uma linha por SKU

df_obs = ler_delta_spark(obs_path)
df_sku = ler_delta_spark(sku_path)
print("features_precos_observacoes:", df_obs.count(), "linhas,", len(df_obs.columns), "colunas")
print("features_precos_sku:", df_sku.count(), "linhas,", len(df_sku.columns), "colunas")

# Colunas do modelo base (as mesmas 7 do notebook 09). As 3 primeiras são categóricas.
FEATURES_BASE = ["id_categoria", "id_categoria_pai", "unidade_cod",
                 "tamanho", "log_tamanho", "tamanho_relativo", "tamanho_rank_pct"]
CATEGORICAS_BASE = ["id_categoria", "id_categoria_pai", "unidade_cod"]
N_ARVORES_BASE = 68   # achado na parada antecipada do notebook 09

pdf = df_obs.toPandas()   # 292.000 linhas cabem tranquilamente na memória

# Treino: folds 1 a 4, só linhas "normais" (usar_no_treino == 1).
treino = pdf[(pdf["conjunto"] == "treino") & (pdf["usar_no_treino"] == 1)].copy()
# Teste: TODAS as linhas do fold 0 (normais e quebradas), porque o alerta é medido nelas.
teste_todas = pdf[pdf["conjunto"] == "teste"].copy()
# Teste normal: só as linhas normais do fold 0, usadas para o MAE.
teste_normais = teste_todas[teste_todas["usar_no_treino"] == 1]

def mae(real, previsto):
    """Erro absoluto médio (em R$): média do tamanho do erro, sem sinal."""
    return float(np.mean(np.abs(real - previsto)))

print("\nLinhas de treino (normais):", len(treino))
print("Linhas de teste (todas):", len(teste_todas), "| normais:", len(teste_normais))
print("SKUs no teste:", teste_todas["sku"].nunique())
print("Rótulos no teste: fraco", int(teste_todas["rotulo_fraco"].sum()),
      "| apoio", int(teste_todas["rotulo_apoio"].sum()))

In [0]:
# ------------------------------------------------------------
# Versão 1 (baseline): modelo base + alerta com MAD calculada no treino
# Treina o LightGBM de 7 colunas (68 árvores), mede o MAE nas linhas normais
# do teste e refaz o alerta do notebook 09 (k = 5). Serve de ponto de comparação
# para as melhorias das próximas etapas.
# Não altera nada no lake.
# ------------------------------------------------------------
# 1) Modelo base: igual ao do notebook 09.
modelo_base = lgb.LGBMRegressor(
    objective="regression_l1", n_estimators=N_ARVORES_BASE, learning_rate=0.05,
    num_leaves=31, min_child_samples=100, random_state=42, verbose=-1,
)
modelo_base.fit(treino[FEATURES_BASE], treino["preco"], categorical_feature=CATEGORICAS_BASE)

# Preço esperado de cada linha do treino e do teste.
treino["previsto"] = modelo_base.predict(treino[FEATURES_BASE])
teste_todas["previsto"] = modelo_base.predict(teste_todas[FEATURES_BASE])
teste_normais = teste_todas[teste_todas["usar_no_treino"] == 1]   # refaz, agora com "previsto"

print("MAE nas linhas normais do teste: R$", round(mae(teste_normais["preco"], teste_normais["previsto"]), 2),
      "(esperado 9,45)")

# 2) Score e z: mesmas contas do notebook 09.
def calcular_score(df):
    """Score = log(preço real / preço esperado). Preço <= 0 vira 0,01 para o log existir."""
    return np.log(np.maximum(df["preco"], 0.01) / df["previsto"])

def calcular_parametros(df_ref):
    """Centro e MAD (bagunça típica) do score, globais e por categoria raiz.

    df_ref: linhas normais usadas como referência (precisa ter 'previsto').
    Retorna um dicionário com os 4 parâmetros usados por calcular_z.
    """
    score = calcular_score(df_ref)
    centro_global = score.median()
    mad_global = (score - centro_global).abs().median()
    por_raiz = score.groupby(df_ref["id_categoria_pai"])
    return {
        "centro_global": centro_global,
        "mad_global": mad_global,
        "centro_raiz": por_raiz.median(),
        "mad_raiz": por_raiz.apply(lambda s: (s - s.median()).abs().median()),
    }

def calcular_z(df, p):
    """Distância do score ao centro do grupo, em unidades de MAD (z). p = parâmetros."""
    centro = df["id_categoria_pai"].map(p["centro_raiz"]).fillna(p["centro_global"])
    mad = df["id_categoria_pai"].map(p["mad_raiz"]).fillna(p["mad_global"])
    return (calcular_score(df) - centro) / (1.4826 * mad)

def avaliar_alertas(df, k):
    """Precision, recall e taxa de alerta (em %) para o limite k."""
    alerta = df["z"].abs() > k
    n_alertas = int(alerta.sum())
    saida = {"k": k, "n_alertas": n_alertas, "taxa_alerta_%": alerta.mean() * 100}
    for rotulo in ["fraco", "apoio"]:
        real = df["rotulo_" + rotulo] == 1
        acertos = int((alerta & real).sum())
        saida["precision_" + rotulo] = acertos / n_alertas * 100 if n_alertas > 0 else np.nan
        saida["recall_" + rotulo] = acertos / int(real.sum()) * 100
    return saida

# 3) Versão 1: MAD calculada nas linhas normais do treino (que o modelo já viu).
params_v1 = calcular_parametros(treino)
print("\nMAD global (v1):", round(params_v1["mad_global"], 3), "(esperado 0,336)")

teste_todas["z_v1"] = calcular_z(teste_todas, params_v1)
tabela_v1 = pd.DataFrame([avaliar_alertas(teste_todas.assign(z=teste_todas["z_v1"]), k)
                          for k in [3, 4, 5, 6, 8, 10]])
print("\n=== Versão 1 no teste (todas as linhas) ===")
print(tabela_v1.round(1).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Erros sintéticos moderados
# Pega as linhas NORMAIS do teste e multiplica o preço de todas por um fator
# (2, 3, 5, 8 ou o inverso), simulando um erro de preço de tamanho conhecido.
# O preço esperado (previsto) não muda, pois o preço não é entrada do modelo.
# Um cenário por fator, mais o original (x1) para medir falsos alarmes.
# Não altera nada no lake.
# ------------------------------------------------------------
FATORES = {
    "÷5": 1 / 5, "÷3": 1 / 3, "÷2": 1 / 2,
    "x1 (original)": 1,
    "x2": 2, "x3": 3, "x5": 5, "x8": 8,
}

def injetar_erro(df, fator):
    """Cópia de df com o preço multiplicado por 'fator'.

    df: linhas normais do teste (já com 'previsto').
    fator: 2 = preço dobrado, 0,5 = preço cortado pela metade, 1 = sem erro.
    """
    saida = df.copy()
    saida["preco"] = saida["preco"] * fator
    saida["fator"] = fator
    return saida

# Conferências antes de injetar: as linhas normais não podem ter rótulo nem preço <= 0.
print("Linhas normais do teste:", len(teste_normais))
print("Soma dos rótulos nessas linhas (esperado 0):",
      int(teste_normais["rotulo_fraco"].sum() + teste_normais["rotulo_apoio"].sum()))
print("Menor preço (esperado > 0): R$", round(teste_normais["preco"].min(), 2))

# Um DataFrame por cenário, já com o z da versão 1.
sinteticos = {}
for nome, fator in FATORES.items():
    df_cenario = injetar_erro(teste_normais, fator)
    df_cenario["z"] = calcular_z(df_cenario, params_v1)
    sinteticos[nome] = df_cenario

# Resumo para conferir se os cenários fazem sentido.
resumo = pd.DataFrame([{
    "cenário": nome,
    "linhas": len(df_c),
    "razão mediana (real/esperado)": float((df_c["preco"] / df_c["previsto"]).median()),
    "z mediano": float(df_c["z"].median()),
} for nome, df_c in sinteticos.items()])
print("\n=== Cenários sintéticos ===")
print(resumo.round(2).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Recall do alerta nos erros sintéticos (versão 1)
# Para cada cenário e cada limite k, calcula a % de linhas com |z| > k.
#   - Nos cenários com erro (÷5 a x8), todas as linhas são erros: a % é o RECALL.
#   - No cenário original (x1), nenhuma linha é erro: a % é a taxa de FALSO ALARME.
# Não altera nada no lake.
# ------------------------------------------------------------
KS_SINTETICO = [2, 3, 4, 5, 6]

tabela_sint_v1 = pd.DataFrame({
    f"k={k}": {nome: (df_c["z"].abs() > k).mean() * 100 for nome, df_c in sinteticos.items()}
    for k in KS_SINTETICO
})
tabela_sint_v1.index.name = "cenário"

print("=== Versão 1: % de linhas em alerta por cenário e limite k ===")
print("(erros: leia como recall | x1 (original): leia como falso alarme)\n")
print(tabela_sint_v1.round(1).to_string())

# Quantas linhas normais (sem erro) entram em alerta, em número absoluto.
original = sinteticos["x1 (original)"]
print("\n=== Falsos alarmes no original (x1): nº de linhas de", len(original), "===")
for k in KS_SINTETICO:
    print(f"k={k}:", int((original['z'].abs() > k).sum()))

In [0]:
# ------------------------------------------------------------
# MAD fora do treino (previsões out-of-fold)
# Para cada fold do treino (1 a 4), treina o modelo base nos outros 3 folds e
# prevê o fold deixado de fora. Como a divisão é por SKU, cada linha do treino
# recebe um preço esperado de um modelo que nunca viu o SKU dela.
# Com essas previsões, recalcula o centro e a MAD (versão 2) e compara com a v1.
# Não altera nada no lake.
# ------------------------------------------------------------
folds_treino = sorted(treino["fold"].unique())
print("Folds do treino:", folds_treino)

treino["previsto_oof"] = np.nan
for f in folds_treino:
    fora = treino["fold"] == f   # linhas do fold deixado de fora
    m = lgb.LGBMRegressor(
        objective="regression_l1", n_estimators=N_ARVORES_BASE, learning_rate=0.05,
        num_leaves=31, min_child_samples=100, random_state=42, verbose=-1,
    )
    m.fit(treino.loc[~fora, FEATURES_BASE], treino.loc[~fora, "preco"],
          categorical_feature=CATEGORICAS_BASE)
    treino.loc[fora, "previsto_oof"] = m.predict(treino.loc[fora, FEATURES_BASE])
    print(f"Fold {f} fora: {int(fora.sum())} linhas | MAE R$",
          round(mae(treino.loc[fora, "preco"], treino.loc[fora, "previsto_oof"]), 2))

print("\nNulos em previsto_oof (esperado 0):", int(treino["previsto_oof"].isna().sum()))
print("MAE no treino com o modelo que já viu as linhas (v1): R$",
      round(mae(treino["preco"], treino["previsto"]), 2))
print("MAE no treino fora do fold (v2):                      R$",
      round(mae(treino["preco"], treino["previsto_oof"]), 2))

# Versão 2: centro e MAD calculados com as previsões fora do fold.
params_v2 = calcular_parametros(treino.assign(previsto=treino["previsto_oof"]))

print("\nMAD global: v1 =", round(params_v1["mad_global"], 3), "| v2 =", round(params_v2["mad_global"], 3))
comparacao_mad = pd.DataFrame({
    "mad_v1": params_v1["mad_raiz"],
    "mad_v2": params_v2["mad_raiz"],
})
comparacao_mad["v2 / v1"] = comparacao_mad["mad_v2"] / comparacao_mad["mad_v1"]
print("\n=== MAD por categoria raiz ===")
print(comparacao_mad.round(3).to_string())

In [0]:
# ------------------------------------------------------------
# Escolha do k com erros sintéticos no TREINO (previsões fora do fold)
# Repete os cenários de erro (÷5 a x8) nas linhas normais do treino, usando o
# preço esperado fora do fold e a MAD da versão 2. A % de linhas em alerta é
# o recall nos cenários com erro e o falso alarme no cenário original (x1).
# O teste NÃO é usado aqui: serve só para confirmar o k escolhido.
# Não altera nada no lake.
# ------------------------------------------------------------
df_oof = treino.assign(previsto=treino["previsto_oof"])   # preço esperado fora do fold

sinteticos_oof = {}
for nome, fator in FATORES.items():
    df_cenario = injetar_erro(df_oof, fator)
    df_cenario["z"] = calcular_z(df_cenario, params_v2)
    sinteticos_oof[nome] = df_cenario

KS_ESCOLHA = [2, 2.5, 3, 3.5, 4, 5]
tabela_oof = pd.DataFrame({
    f"k={k}": {nome: (df_c["z"].abs() > k).mean() * 100 for nome, df_c in sinteticos_oof.items()}
    for k in KS_ESCOLHA
})
tabela_oof.index.name = "cenário"

print("=== Treino fora do fold, MAD v2: % de linhas em alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_oof.round(2).to_string())

original_oof = sinteticos_oof["x1 (original)"]
print("\n=== Falsos alarmes no original (x1): nº de linhas de", len(original_oof), "===")
for k in KS_ESCOLHA:
    print(f"k={k}:", int((original_oof['z'].abs() > k).sum()),
          f"({(original_oof['z'].abs() > k).mean() * 100:.3f}%)")

In [0]:
# ------------------------------------------------------------
# Confirmação no teste: versão 1 (k = 5) contra versão 2 (k = 3 e k = 3,5)
# 1) Contra os rótulos reais (fraco e apoio): alertas, precision e recall.
# 2) Nos erros sintéticos do teste: % de linhas em alerta por cenário.
# O k foi escolhido no treino (célula 9); o teste só confirma.
# Não altera nada no lake.
# ------------------------------------------------------------
K_V1 = 5      # baseline
K_V2 = 3      # escolhido pelo critério: menor k com falso alarme <= 0,05% no treino
K_V2_CAUT = 3.5   # alternativa mais conservadora (0 falso alarme no treino)

teste_todas["z_v2"] = calcular_z(teste_todas, params_v2)

versoes = {
    f"v1 (k={K_V1})": ("z_v1", K_V1),
    f"v2 (k={K_V2})": ("z_v2", K_V2),
    f"v2 (k={K_V2_CAUT})": ("z_v2", K_V2_CAUT),
}

# 1) Contra os rótulos reais.
linhas = []
for nome, (coluna_z, k) in versoes.items():
    r = avaliar_alertas(teste_todas.assign(z=teste_todas[coluna_z]), k)
    r["versão"] = nome
    linhas.append(r)
tabela_rotulos = pd.DataFrame(linhas).set_index("versão").drop(columns="k")
print("=== Teste (", len(teste_todas), "linhas): alertas contra os rótulos reais ===")
print(tabela_rotulos.round(1).to_string())

# 2) Nos erros sintéticos do teste (todas as linhas normais com o preço multiplicado).
resultado = {}
for cenario, df_c in sinteticos.items():
    z_v1 = df_c["z"]                       # já calculado com a MAD v1
    z_v2 = calcular_z(df_c, params_v2)     # MAD v2
    resultado[cenario] = {
        f"v1 (k={K_V1})": (z_v1.abs() > K_V1).mean() * 100,
        f"v2 (k={K_V2})": (z_v2.abs() > K_V2).mean() * 100,
        f"v2 (k={K_V2_CAUT})": (z_v2.abs() > K_V2_CAUT).mean() * 100,
    }
tabela_sint_teste = pd.DataFrame(resultado).T
tabela_sint_teste.index.name = "cenário"
print("\n=== Teste, erros sintéticos: % de linhas em alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_sint_teste.round(1).to_string())

In [0]:
# ------------------------------------------------------------
# Exemplos concretos
# 1) Alertas extras da versão 2 (k = 3) no teste: linhas em alerta que nenhum
#    rótulo marcou. Compara o preço com o esperado e com a mediana do próprio SKU.
# 2) Cinco exemplos de erro sintético (x5) que a v1 (k = 5) não pegava e a v2 pega.
# Só lê os dados já carregados. Não altera nada no lake.
# ------------------------------------------------------------
# 1) Alertas extras do teste.
extras = teste_todas[
    (teste_todas["z_v2"].abs() > K_V2)
    & (teste_todas["rotulo_fraco"] == 0)
    & (teste_todas["rotulo_apoio"] == 0)
].copy()
extras["razao_esperado"] = extras["preco"] / extras["previsto"]
extras["razao_mediana_sku"] = extras["preco"] / extras["mediana_sku"]
extras = extras.reindex(extras["z_v2"].abs().sort_values(ascending=False).index)

colunas = ["sku", "id_categoria", "tamanho", "preco", "previsto", "mediana_sku",
           "razao_esperado", "razao_mediana_sku", "z_v1", "z_v2"]
print("=== Alertas extras da v2 (k =", K_V2, ") no teste: ", len(extras), "linhas ===")
print(extras[colunas].round(2).to_string(index=False))

# 2) Exemplos sintéticos: x5 que a v1 não pega (|z| <= 5) e a v2 pega (|z| > 3).
df_x5 = sinteticos["x5"].copy()
df_x5["z_v2"] = calcular_z(df_x5, params_v2)
df_x5["preco_original"] = df_x5["preco"] / 5
df_x5["razao_esperado"] = df_x5["preco"] / df_x5["previsto"]
exemplos = df_x5[(df_x5["z"].abs() <= K_V1) & (df_x5["z_v2"].abs() > K_V2)]
print("\nLinhas x5 que a v1 perde e a v2 pega:", len(exemplos), "de", len(df_x5))
print(exemplos.sample(5, random_state=42)[
    ["sku", "id_categoria", "tamanho", "preco_original", "preco", "previsto",
     "razao_esperado", "z", "z_v2"]
].rename(columns={"preco": "preco_com_erro", "z": "z_v1"}).round(2).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Tabela final: evolução do baseline (v1 contra v2)
# Junta, para o teste: a MAD, o limite k, os alertas, precision e recall contra
# os rótulos reais, o recall nos erros sintéticos e o falso alarme nas linhas normais.
# Só usa resultados já calculados nas células anteriores. Não altera nada no lake.
# ------------------------------------------------------------
versoes_final = {
    "v1 (k=5)":   (params_v1, "z_v1", 5),
    "v2 (k=3)":   (params_v2, "z_v2", 3),
    "v2 (k=3.5)": (params_v2, "z_v2", 3.5),
}
mae_normais = mae(teste_normais["preco"], teste_normais["previsto"])

colunas_resumo = {}
for nome, (p, coluna_z, k) in versoes_final.items():
    alerta = teste_todas[coluna_z].abs() > k
    fora_dos_rotulos = alerta & (teste_todas["rotulo_fraco"] == 0) & (teste_todas["rotulo_apoio"] == 0)
    r = avaliar_alertas(teste_todas.assign(z=teste_todas[coluna_z]), k)

    d = {
        "MAE linhas normais (R$)": mae_normais,
        "MAD global": p["mad_global"],
        "limite k": k,
        "alertas no teste": r["n_alertas"],
        "alertas fora dos rótulos": int(fora_dos_rotulos.sum()),
        "precision apoio (%)": r["precision_apoio"],
        "recall apoio (%)": r["recall_apoio"],
        "precision fraco (%)": r["precision_fraco"],
        "recall fraco (%)": r["recall_fraco"],
    }
    for cenario in ["÷5", "÷3", "÷2", "x2", "x3", "x5", "x8"]:
        d[f"recall {cenario} (%)"] = tabela_sint_teste.loc[cenario, nome]
    d["falso alarme em linhas normais (%)"] = tabela_sint_teste.loc["x1 (original)", nome]
    colunas_resumo[nome] = d

resumo_evolucao = pd.DataFrame(colunas_resumo)
print("=== Evolução do baseline (teste: 765 SKUs que o modelo não viu) ===")
print(resumo_evolucao.round(3).to_string())

In [0]:
# ------------------------------------------------------------
# Dois níveis de alerta (proposta aprovada pelo Geovany)
# Classifica cada linha em "forte" (|z| acima de 5), "revisão" (|z| entre 3 e 5)
# ou "normal", usando o z da versão 2 (MAD fora do treino).
# Mostra a contagem no teste, o cruzamento com os rótulos reais e a % de linhas
# em cada nível nos erros sintéticos.
# Só lê os dados já carregados. Não altera nada no lake.
# ------------------------------------------------------------
LIMITE_FORTE = 5
LIMITE_REVISAO = K_V2   # 3, escolhido no treino (célula 9)
ORDEM = ["forte", "revisão", "normal"]

def classificar_nivel(z):
    """Nível do alerta a partir do z: 'forte', 'revisão' ou 'normal'.

    z: Series com o z de cada linha.
    """
    z_abs = z.abs()
    nivel = np.select([z_abs > LIMITE_FORTE, z_abs > LIMITE_REVISAO],
                      ["forte", "revisão"], default="normal")
    return pd.Series(nivel, index=z.index)

teste_todas["nivel_alerta"] = classificar_nivel(teste_todas["z_v2"])

# 1) Contagem no teste e cruzamento com os rótulos reais.
def contar(df):
    return df["nivel_alerta"].value_counts().reindex(ORDEM, fill_value=0)

sem_rotulo = (teste_todas["rotulo_fraco"] == 0) & (teste_todas["rotulo_apoio"] == 0)
tabela_nivel = pd.DataFrame({
    "linhas": contar(teste_todas),
    "no rótulo fraco": contar(teste_todas[teste_todas["rotulo_fraco"] == 1]),
    "no rótulo de apoio": contar(teste_todas[teste_todas["rotulo_apoio"] == 1]),
    "sem rótulo": contar(teste_todas[sem_rotulo]),
})
print("=== Teste (", len(teste_todas), "linhas): alertas por nível ===")
print(tabela_nivel.to_string())

# Conferência: forte + revisão tem que bater com o alerta da v2 com k = 3.
total_alertas = int((teste_todas["nivel_alerta"] != "normal").sum())
print("\nForte + revisão:", total_alertas,
      "| alertas da v2 com k = 3:", int((teste_todas["z_v2"].abs() > K_V2).sum()))

# 2) Erros sintéticos do teste: % de linhas em cada nível.
linhas_sint = []
for cenario, df_c in sinteticos.items():
    nivel = classificar_nivel(calcular_z(df_c, params_v2))
    linhas_sint.append({
        "cenário": cenario,
        "forte (%)": (nivel == "forte").mean() * 100,
        "revisão (%)": (nivel == "revisão").mean() * 100,
        "total em alerta (%)": (nivel != "normal").mean() * 100,
    })
tabela_nivel_sint = pd.DataFrame(linhas_sint).set_index("cenário")
print("\n=== Teste, erros sintéticos: % de linhas por nível de alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_nivel_sint.round(1).to_string())

In [0]:
# ------------------------------------------------------------
# Evolução 6 (preparação): linhas e SKUs de treino por categoria
# Antes de calcular a MAD por categoria (em vez de por categoria raiz), conta
# quanto dado cada categoria tem no treino. Categorias com pouco dado teriam
# uma MAD instável e vão precisar de uma regra de segurança (voltar à MAD da raiz).
# Só lê os dados já carregados. Não altera nada no lake.
# ------------------------------------------------------------
contagem = (
    treino.groupby("id_categoria")
    .agg(linhas_treino=("preco", "size"),
         skus_treino=("sku", "nunique"),
         n_raizes=("id_categoria_pai", "nunique"))
)
linhas_teste = teste_todas.groupby("id_categoria").size().rename("linhas_teste")
contagem = contagem.join(linhas_teste, how="outer").fillna(0)
total_teste = contagem["linhas_teste"].sum()

sem_treino = contagem["linhas_treino"] == 0
print("Categorias (id_categoria) no treino:", int((~sem_treino).sum()))
print("Categorias raiz (id_categoria_pai) no treino:", treino["id_categoria_pai"].nunique())
print("Categorias do teste que não aparecem no treino:",
      int((sem_treino & (contagem["linhas_teste"] > 0)).sum()),
      "| linhas de teste nelas:", int(contagem.loc[sem_treino, "linhas_teste"].sum()))
print("Categorias ligadas a mais de uma raiz (esperado 0):", int((contagem["n_raizes"] > 1).sum()))

print("\n=== Linhas e SKUs de treino por categoria ===")
print(contagem.loc[~sem_treino, ["linhas_treino", "skus_treino"]].describe().round(0).to_string())

def tabela_cortes(coluna, cortes, nome):
    """Para cada mínimo, quantas categorias ficam abaixo e que % das linhas de teste elas têm."""
    linhas = []
    for c in cortes:
        abaixo = contagem[coluna] < c
        linhas.append({
            nome: c,
            "categorias abaixo": int(abaixo.sum()),
            "% das categorias": abaixo.mean() * 100,
            "% das linhas do teste nelas": contagem.loc[abaixo, "linhas_teste"].sum() / total_teste * 100,
        })
    return pd.DataFrame(linhas)

print("\n=== Categorias abaixo de cada mínimo de SKUs no treino ===")
print(tabela_cortes("skus_treino", [5, 10, 20, 30, 50], "mínimo de SKUs").round(1).to_string(index=False))
print("\n=== Categorias abaixo de cada mínimo de linhas no treino ===")
print(tabela_cortes("linhas_treino", [30, 100, 300, 1000], "mínimo de linhas").round(1).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Evolução 6: MAD por categoria (com regra de segurança), escolhida no treino
# Compara "só raiz" (versão 2) com MAD por categoria usando 5, 10 e 20 SKUs como
# mínimo. Categorias com SKUs abaixo do mínimo voltam à MAD da categoria raiz.
# Para ser justo, o centro e a MAD de cada fold do treino vêm dos OUTROS folds
# (SKUs diferentes), como acontece no teste. O teste NÃO é usado aqui.
# Critério fixado antes dos números: menor k com falso alarme <= 0,05% no treino;
# vence a configuração com maior recall médio nos erros sintéticos.
# Não altera nada no lake.
# ------------------------------------------------------------
def parametros_cat(df_ref, min_skus):
    """Centro e MAD do score: globais, por raiz e por categoria (só com SKUs suficientes).

    df_ref: linhas normais de referência (com 'previsto' fora do fold).
    min_skus: mínimo de SKUs distintos para a categoria ter centro e MAD próprios.
    """
    p = calcular_parametros(df_ref)   # global e por raiz (função já existente)
    score = calcular_score(df_ref)
    skus_por_cat = df_ref.groupby("id_categoria")["sku"].nunique()
    cats_ok = skus_por_cat[skus_por_cat >= min_skus].index
    em_cat_ok = df_ref["id_categoria"].isin(cats_ok)

    if em_cat_ok.any():
        por_cat = score[em_cat_ok].groupby(df_ref.loc[em_cat_ok, "id_categoria"])
        p["centro_cat"] = por_cat.median()
        p["mad_cat"] = por_cat.apply(lambda s: (s - s.median()).abs().median())
    else:
        p["centro_cat"] = pd.Series(dtype="float64")
        p["mad_cat"] = pd.Series(dtype="float64")
    return p

def calcular_z_cat(df, p):
    """z do score: usa centro e MAD da categoria; se não houver, da raiz; se não, global."""
    centro = (df["id_categoria"].map(p["centro_cat"])
              .fillna(df["id_categoria_pai"].map(p["centro_raiz"]))
              .fillna(p["centro_global"]))
    mad = (df["id_categoria"].map(p["mad_cat"])
           .fillna(df["id_categoria_pai"].map(p["mad_raiz"]))
           .fillna(p["mad_global"]))
    return (calcular_score(df) - centro) / (1.4826 * mad)

CONFIGURACOES = {
    "só raiz (v2)": 10**9,          # nenhuma categoria atinge o mínimo: só a MAD da raiz
    "categoria, mín. 5 SKUs": 5,
    "categoria, mín. 10 SKUs": 10,
    "categoria, mín. 20 SKUs": 20,
}
KS_NIVEL = [2, 2.5, 3, 3.5, 4, 5]
LIMITE_FALSO_ALARME = 0.05   # % de linhas normais em alerta
CENARIOS_ERRO = ["÷5", "÷3", "÷2", "x2", "x3", "x5", "x8"]

linhas = []
for nome, min_skus in CONFIGURACOES.items():
    # Parâmetros de cada fold vêm dos outros 3 folds do treino.
    params_folds = {
        f: parametros_cat(df_oof[df_oof["fold"] != f], min_skus) for f in folds_treino
    }
    # |z| das linhas do treino em cada cenário (preço multiplicado pelo fator).
    z_cenario = {}
    for cenario, fator in FATORES.items():
        partes = []
        for f in folds_treino:
            alvo = injetar_erro(df_oof[df_oof["fold"] == f], fator)
            partes.append(calcular_z_cat(alvo, params_folds[f]).abs())
        z_cenario[cenario] = pd.concat(partes)

    falso = {k: (z_cenario["x1 (original)"] > k).mean() * 100 for k in KS_NIVEL}
    aceitos = [k for k in KS_NIVEL if falso[k] <= LIMITE_FALSO_ALARME]
    linha = {
        "configuração": nome,
        "categorias com MAD própria": int((contagem["skus_treino"] >= min_skus).sum()),
    }
    if aceitos:
        k = min(aceitos)
        linha["k escolhido"] = k
        linha["falso alarme (%)"] = falso[k]
        for cenario in CENARIOS_ERRO:
            linha[f"recall {cenario} (%)"] = (z_cenario[cenario] > k).mean() * 100
        linha["recall médio (%)"] = np.mean([linha[f"recall {c} (%)"] for c in CENARIOS_ERRO])
    else:
        linha["k escolhido"] = np.nan
    linhas.append(linha)

tabela_escolha_cat = pd.DataFrame(linhas).set_index("configuração")
print("=== Treino (MAD de outros folds): k escolhido e recall por configuração ===")
print("(critério: menor k com falso alarme <= 0,05%; erros: recall | x1: falso alarme)\n")
print(tabela_escolha_cat.round(3).T.to_string())

### Evolução 6: MAD por categoria (testada e descartada)

**Antes:** MAD e centro por categoria raiz (12 grupos), com k = 3 escolhido no treino. No treino, com a MAD vinda de outros folds: falso alarme de 0,037% e recall médio de 35,3% nos erros sintéticos (58,3% em ÷5, 57,3% em ×5 e 81,6% em ×8).

**O que foi alterado:** MAD e centro por categoria (123 grupos) em vez de por categoria raiz. Categorias com menos SKUs que o mínimo (5, 10 ou 20) voltam à MAD da raiz. O k foi escolhido de novo, pelo mesmo critério (falso alarme de no máximo 0,05%). Tudo medido no treino, com centro e MAD calculados nos outros folds (SKUs diferentes), sem usar o teste.

**Depois:**

| Configuração | Categorias com MAD própria | k escolhido | Recall ÷5 | Recall ×5 | Recall ×8 | Recall médio |
|---|---|---|---|---|---|---|
| Só raiz (v2) | 0 | 3 | 58,3% | 57,3% | 81,6% | 35,3% |
| Categoria, mín. 5 SKUs | 87 | 3,5 | 41,1% | 38,4% | 69,5% | 24,3% |
| Categoria, mín. 10 SKUs | 73 | 3,5 | 41,1% | 38,5% | 69,6% | 24,3% |
| Categoria, mín. 20 SKUs | 54 | 3,5 | 41,5% | 39,1% | 69,9% | 24,6% |

**Ganho:** nenhum. Com a MAD por categoria, o k = 3 deixa de respeitar o limite de falso alarme, o k sobe para 3,5 e o recall médio cai cerca de 11 pontos percentuais, qualquer que seja o mínimo de SKUs.

**Decisão:** manter a MAD por categoria raiz. A MAD do alerta não é o fator que limita o resultado. As próximas evoluções devem mudar o que o modelo recebe, e não o cálculo do limite.

In [0]:
# Conferência: colunas da tabela de observações e da tabela de SKUs.
# Só lê. Não altera nada.
resumo_colunas = pd.DataFrame({
    "tipo": pdf.dtypes.astype(str),
    "valores distintos": pdf.nunique(),
    "exemplo": pdf.iloc[0],
    "usada no modelo": pdf.columns.isin(FEATURES_BASE),
})
print("=== Colunas de features_precos_observacoes (pdf) ===")
print(resumo_colunas.to_string())
print("\n=== Colunas de features_precos_sku (df_sku) ===")
print(df_sku.columns)

In [0]:
# ------------------------------------------------------------
# Diagnóstico: o preço varia menos dentro do mesmo SKU do que entre SKUs?
# Compara o preço com a mediana do próprio SKU (coluna mediana_sku) e mede,
# no treino, a variação desse score e o recall nos erros sintéticos, com os
# mesmos fatores (÷5 a x8) das células anteriores.
# Só lê os dados já carregados. Não treina modelo e não altera nada no lake.
# ------------------------------------------------------------
t = treino.copy()
print("Observações por SKU no treino:")
print(t.groupby("sku").size().describe().round(0).to_string())

def score_sku(df, fator=1):
    """log(preço com erro / mediana do próprio SKU)."""
    return np.log(np.maximum(df["preco"] * fator, 0.01) / df["mediana_sku"])

s0 = score_sku(t)
centro_sku = s0.median()
mad_sku = (s0 - centro_sku).abs().median()
print("\nMAD do score pelo histórico do SKU:", round(mad_sku, 3),
      "| MAD do modelo (v2):", round(params_v2["mad_global"], 3))

KS_DIAG = [3, 3.5, 5]
tabela_diag = pd.DataFrame({
    f"k={k}": {nome: (((score_sku(t, fator) - centro_sku).abs() / (1.4826 * mad_sku)) > k).mean() * 100
               for nome, fator in FATORES.items()}
    for k in KS_DIAG
})
tabela_diag.index.name = "cenário"
print("\n=== Treino: % de linhas em alerta usando o histórico do próprio SKU ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_diag.round(2).to_string())

### Diagnóstico: variação do preço dentro do mesmo SKU

**O que foi medido:** comparamos o preço de cada linha do treino com a mediana do próprio SKU (coluna `mediana_sku`), em vez de com o preço esperado pelo modelo, e medimos a MAD desse score e o recall nos erros sintéticos (÷5 a ×8). Cada SKU tem em média 75 observações no treino (mínimo de 41 e máximo de 122).

**Resultado:**

| Método (treino) | MAD do score | Recall ÷5 | Recall ×5 | Recall ×8 | Falso alarme |
|---|---|---|---|---|---|
| LightGBM v2 (k = 3, MAD de outros folds) | 0,338 | 58,3% | 57,3% | 81,6% | 0,037% |
| Histórico do próprio SKU (k = 3) | 0,324 | 64,9% | 62,2% | 84,8% | 0,05% |

**Leitura:** mesmo conhecendo a mediana do próprio SKU, o preço varia cerca de 32% entre observações (MAD de 0,324, contra 0,338 do modelo). O modelo está perto do que as colunas atuais permitem: o histórico do SKU aumenta o recall em 3 a 6 pontos percentuais, com o falso alarme no limite. Isso é coerente com o resultado negativo da evolução 6 e com o empate com a regra simples de mediana por subcategoria e tamanho.

**Limitações:** a mediana do SKU inclui a própria linha, e o LightGBM foi medido com a MAD de outros folds, então o histórico do SKU sai um pouco favorecido. A comparação é feita só no treino, com erros sintéticos. Um SKU novo não tem histórico, e usar essa informação exigiria uma decisão do negócio.

**Decisão:** não seguir com o histórico do SKU como evolução do LightGBM. Para melhorar o alerta é preciso informação que explique por que o preço muda de uma observação para outra do mesmo SKU. O próximo passo é verificar quais colunas por observação existem na tabela e ainda não entram no modelo.

In [0]:
# ------------------------------------------------------------
# Diagnóstico: a data da observação explica parte da variação do preço?
# Extrai a data do caminho do arquivo de origem (bronze_source_file) e mede,
# no treino, 1) quanto do score do modelo é um efeito comum da data (centro de
# cada data calculado nos OUTROS folds) e 2) quanto o preço varia em relação
# à observação anterior do mesmo SKU.
# Só lê os dados já carregados. Não treina modelo e não altera nada no lake.
# ------------------------------------------------------------
padrao_data = r"real-time-data/(\d{4})/(\d{2})/(\d{2})/"

d = df_oof.copy()   # 'previsto' = previsão fora do fold
partes = d["bronze_source_file"].str.extract(padrao_data)
d["data_obs"] = pd.to_datetime(partes[0] + "-" + partes[1] + "-" + partes[2], errors="coerce")

print("Linhas sem data reconhecida (esperado 0):", int(d["data_obs"].isna().sum()))
print("Período:", d["data_obs"].min().date(), "a", d["data_obs"].max().date(),
      "| datas distintas:", d["data_obs"].nunique())
print("Linhas por data no treino:")
print(d.groupby("data_obs").size().describe().round(0).to_string())

def mad(s):
    """Mediana dos desvios absolutos em relação à mediana."""
    return (s - s.median()).abs().median()

# 1) Efeito da data: centro de cada data calculado nos outros folds.
d["score"] = calcular_score(d)
d["centro_data"] = np.nan
for f in folds_treino:
    fora = d["fold"] == f
    centros = d.loc[~fora].groupby("data_obs")["score"].median()
    d.loc[fora, "centro_data"] = d.loc[fora, "data_obs"].map(centros)
d["centro_data"] = d["centro_data"].fillna(d["score"].median())

residuo = d["score"] - d["centro_data"]
print("\nMAD do score do modelo:                        ", round(mad(d["score"]), 3))
print("MAD depois de tirar o efeito da data:          ", round(mad(residuo), 3))
print("Variação entre as datas (MAD dos centros):     ", round(mad(d.groupby("data_obs")["score"].median()), 3))

# 2) Variação em relação à observação anterior do mesmo SKU.
d = d.sort_values(["sku", "data_obs"])
d["preco_anterior"] = d.groupby("sku")["preco"].shift(1)
variacao = np.log(np.maximum(d["preco"], 0.01) / np.maximum(d["preco_anterior"], 0.01)).dropna()
print("\nMAD de log(preço / preço anterior do mesmo SKU):", round(mad(variacao), 3),
      "| linhas:", len(variacao))
print("(se as observações fossem independentes, esperaríamos algo perto de 0,46)")

# 3) Recall nos erros sintéticos com o efeito da data retirado do score.
c0 = residuo.median()
mad0 = mad(residuo)
KS_DATA = [3, 3.5, 5]
tabela_data = pd.DataFrame({
    f"k={k}": {
        nome: (((np.log(np.maximum(d["preco"] * fator, 0.01) / d["previsto"])
                 - d["centro_data"] - c0).abs() / (1.4826 * mad0)) > k).mean() * 100
        for nome, fator in FATORES.items()
    } for k in KS_DATA
})
tabela_data.index.name = "cenário"
print("\n=== Treino: % de linhas em alerta com o efeito da data retirado ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_data.round(2).to_string())

In [0]:
# ------------------------------------------------------------
# Diagnóstico: os erros rotulados ficam concentrados em poucos SKUs?
# Conta, para cada SKU com alguma linha de rótulo de apoio, quantas linhas
# rotuladas ele tem e que % das observações dele isso representa.
# Só lê os dados já carregados. Não altera nada no lake.
# ------------------------------------------------------------
rot = pdf[pdf["rotulo_apoio"] == 1]
por_sku = rot.groupby("sku").size().rename("linhas_rotuladas").to_frame()
por_sku["observacoes_do_sku"] = pdf.groupby("sku").size().reindex(por_sku.index)
por_sku["% do SKU rotulado"] = por_sku["linhas_rotuladas"] / por_sku["observacoes_do_sku"] * 100

print("Linhas com rótulo de apoio:", len(rot),
      "| SKUs afetados:", len(por_sku), "de", pdf["sku"].nunique())
print(por_sku.describe().round(1).to_string())

### Diagnóstico: data da observação e persistência do preço

**O que foi medido:** extraímos a data do caminho do arquivo de origem (`bronze_source_file`) e medimos, no treino, (1) quanto do score do modelo é um efeito comum da data (centro de cada data calculado nos outros folds), (2) quanto o preço varia em relação à observação anterior do mesmo SKU e (3) se os erros reais rotulados se concentram em poucos SKUs. As observações vão de 01 a 15 de outubro de 2026, com 6 datas distintas e 365 arquivos de origem (cerca de 60 capturas por dia).

**Resultado:**

| Medida | Resultado |
|---|---|
| MAD do score do modelo | 0,338 |
| MAD depois de tirar o efeito da data | 0,337 |
| Variação entre as datas (MAD dos centros) | 0,005 |
| MAD de log(preço / preço anterior do mesmo SKU) | 0,423 (225.197 linhas) |
| Recall com a data retirada (k = 3) | ÷5: 58,2%; ×5: 57,5%; ×8: 81,9% |
| Linhas com rótulo de apoio | 6.551, em 2.946 dos 3.805 SKUs |
| Linhas rotuladas por SKU afetado | mediana de 2 (2,5% das observações do SKU); máximo de 13 (17,5%) |

**Leitura:** a data não explica a variação do preço (a MAD quase não muda e as datas praticamente não se diferenciam). A variação em relação à observação anterior do mesmo SKU (0,423) fica perto do que se esperaria de observações independentes (cerca de 0,46): o preço de um SKU num momento quase não prevê o do momento seguinte. Os erros reais rotulados são pontuais, espalhados por muitos SKUs e sem concentração em poucos. O recall continua igual ao do LightGBM atual.

**Limitações:** a comparação é feita só no treino, com erros sintéticos que assumem erro isolado em uma observação. Os rótulos marcam erros extremos (razão preço real / esperado menor ou igual a 0 ou maior ou igual a 14), e não sabemos se erros moderados reais também são pontuais.

**Decisão:** não usar a data como atributo nem como correção do score, e manter o alerta linha a linha. Juntar observações do mesmo SKU para reduzir o ruído não serve, porque os erros reais são pontuais e seriam diluídos. O próximo passo é verificar se o ruído é diferente de um SKU para outro, o que permitiria um limite por SKU.

In [0]:
# ------------------------------------------------------------
# Evolução 7: limite por SKU (centro e MAD do próprio SKU)
# Separa as observações de cada SKU em duas metades aleatórias: o centro e a MAD
# vêm de uma metade e o alerta é medido na outra (nos dois sentidos), para o limite
# nunca ser avaliado nas linhas que o calcularam. SKUs com poucas observações na
# metade de referência (menos de MIN_OBS) voltam à MAD da raiz.
# Compara: só raiz (v2), SKU puro e SKU misturado com a raiz (média simples da MAD).
# Critério fixado antes dos números: menor k com falso alarme <= 0,05% no treino;
# vence a versão com maior recall médio nos erros sintéticos. O teste NÃO é usado.
# Não altera nada no lake.
# ------------------------------------------------------------
MIN_OBS = 20
KS_SKU = [2, 2.5, 3, 3.5, 4, 5]
LIMITE_FALSO_ALARME = 0.05
CENARIOS_ERRO = ["÷5", "÷3", "÷2", "x2", "x3", "x5", "x8"]

rng = np.random.default_rng(42)
d = df_oof.copy()   # 'previsto' = previsão fora do fold
d["score"] = calcular_score(d)
d["metade"] = rng.integers(0, 2, len(d))

def mad(s):
    """Mediana dos desvios absolutos em relação à mediana."""
    return (s - s.median()).abs().median()

# Heterogeneidade do ruído: a MAD de um SKU se repete entre as duas metades?
mad_metades = d.groupby(["sku", "metade"])["score"].apply(mad).unstack()
print("MAD do score por SKU (metade 0), percentis:")
print(mad_metades[0].describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())
print("\nCorrelação (Spearman) da MAD do SKU entre as duas metades:",
      round(mad_metades[0].corr(mad_metades[1], method="spearman"), 3))

def z_sku(df_alvo_f, centro_sku, mad_sku, n_sku, mistura):
    """z com centro e MAD do próprio SKU; mistura = peso da MAD do SKU (1 = só SKU)."""
    score_f = calcular_score(df_alvo_f)
    c_raiz = (df_alvo_f["id_categoria_pai"].map(params_v2["centro_raiz"])
              .fillna(params_v2["centro_global"]))
    m_raiz = (df_alvo_f["id_categoria_pai"].map(params_v2["mad_raiz"])
              .fillna(params_v2["mad_global"]))
    usa = df_alvo_f["sku"].map(n_sku).fillna(0) >= MIN_OBS
    c = df_alvo_f["sku"].map(centro_sku).where(usa, c_raiz)
    m_sku = df_alvo_f["sku"].map(mad_sku).where(usa, m_raiz)
    m = mistura * m_sku + (1 - mistura) * m_raiz
    return (score_f - c) / (1.4826 * m)

versoes = ["só raiz (v2)", "SKU puro", "SKU misturado com a raiz"]
z_todas = {v: {c: [] for c in FATORES} for v in versoes}

for h in (0, 1):
    ref = d[d["metade"] == h]
    alvo = d[d["metade"] != h]
    grupo = ref.groupby("sku")["score"]
    centro_sku = grupo.median()
    mad_sku = grupo.apply(mad)
    n_sku = grupo.size()
    for cenario, fator in FATORES.items():
        alvo_f = injetar_erro(alvo, fator)
        z_todas["só raiz (v2)"][cenario].append(calcular_z(alvo_f, params_v2).abs())
        z_todas["SKU puro"][cenario].append(z_sku(alvo_f, centro_sku, mad_sku, n_sku, 1.0).abs())
        z_todas["SKU misturado com a raiz"][cenario].append(
            z_sku(alvo_f, centro_sku, mad_sku, n_sku, 0.5).abs())

linhas = []
for v in versoes:
    z = {c: pd.concat(z_todas[v][c]) for c in FATORES}
    falso = {k: (z["x1 (original)"] > k).mean() * 100 for k in KS_SKU}
    aceitos = [k for k in KS_SKU if falso[k] <= LIMITE_FALSO_ALARME]
    linha = {"versão": v}
    if aceitos:
        k = min(aceitos)
        linha["k escolhido"] = k
        linha["falso alarme (%)"] = falso[k]
        for c in CENARIOS_ERRO:
            linha[f"recall {c} (%)"] = (z[c] > k).mean() * 100
        linha["recall médio (%)"] = np.mean([linha[f"recall {c} (%)"] for c in CENARIOS_ERRO])
    else:
        linha["k escolhido"] = np.nan
    linhas.append(linha)

tabela_sku = pd.DataFrame(linhas).set_index("versão")
print("\n=== Treino (metade de referência x metade avaliada): k escolhido e recall ===")
print("(critério: menor k com falso alarme <= 0,05%; erros: recall | x1: falso alarme)\n")
print(tabela_sku.round(3).T.to_string())

### Evolução 7: limite por SKU (testada e descartada)

**Antes:** centro e MAD por categoria raiz, com k = 3. Na mesma avaliação (metade das observações de cada SKU como referência e a outra metade como teste, sem usar o teste real): falso alarme de 0,032% e recall médio de 35,3% nos erros sintéticos (58,3% em ÷5, 57,3% em ×5 e 81,5% em ×8).

**O que foi alterado:** centro e MAD calculados com as observações do próprio SKU, em duas versões: SKU puro e SKU misturado com a raiz (média simples das duas MAD). SKUs com menos de 20 observações na metade de referência voltam à MAD da raiz. O k foi escolhido pelo mesmo critério de antes (falso alarme de no máximo 0,05%).

**Depois:**

| Versão | k escolhido | Falso alarme | Recall ÷5 | Recall ×5 | Recall ×8 | Recall médio |
|---|---|---|---|---|---|---|
| Só raiz (v2) | 3 | 0,032% | 58,3% | 57,3% | 81,5% | 35,3% |
| SKU puro | nenhum k da grade (2 a 5) atinge o limite de falso alarme | n/d | n/d | n/d | n/d | n/d |
| SKU misturado com a raiz | 5 | 0,014% | 11,7% | 1,8% | 21,0% | 5,3% |

n/d = não disponível, porque nenhum k da grade chegou ao limite de falso alarme.

O ruído por SKU varia pouco: a MAD do score por SKU tem mediana de 0,32, vai de 0,19 (5%) a 0,47 (95%), e a correlação entre as duas metades do mesmo SKU é de 0,26.

**Ganho:** nenhum. Com o limite por SKU o recall médio cai de 35,3% para 5,3%.

**Decisão:** manter a MAD por categoria raiz. A diferença de ruído entre SKUs é em grande parte acaso da amostra e não uma característica estável do SKU, por isso estimar um limite por SKU com cerca de 37 observações só acrescenta incerteza.

In [0]:
# Diagnóstico: a MAD do score muda com o nível de preço esperado ou com a unidade?
# Só lê os dados já carregados. Não altera nada.
d = df_oof.copy()   # 'previsto' = previsão fora do fold
d["score"] = calcular_score(d)
d["faixa_preco"] = pd.qcut(d["previsto"], 10, duplicates="drop")

def mad(s):
    """Mediana dos desvios absolutos em relação à mediana."""
    return (s - s.median()).abs().median()

por_faixa = d.groupby("faixa_preco", observed=True)["score"].agg(
    linhas="size", centro="median", MAD=mad)
por_unidade = d.groupby("unidade_cod")["score"].agg(
    linhas="size", centro="median", MAD=mad)

print("=== MAD do score por faixa de preço esperado (10 faixas) ===")
print(por_faixa.round(3).to_string())
print("\n=== MAD do score por unidade ===")
print(por_unidade.round(3).to_string())

In [0]:
# ------------------------------------------------------------
# Evolução 8: centro e MAD por faixa de preço esperado
# Divide as linhas do treino em faixas pelo preço esperado (previsto fora do fold)
# e calcula o centro e a MAD do score em cada faixa, em vez de por categoria raiz.
# Os parâmetros de cada fold do treino vêm dos OUTROS folds (SKUs diferentes).
# Compara: só raiz (v2) contra 5, 10 e 20 faixas.
# Critério fixado antes dos números: menor k com falso alarme <= 0,05% no treino;
# vence a versão com maior recall médio nos erros sintéticos. O teste NÃO é usado.
# Não altera nada no lake.
# ------------------------------------------------------------
def cortes_faixas(previsto, n_faixas):
    """Limites das faixas (quantis do preço esperado), abertos nas duas pontas."""
    cortes = np.quantile(previsto, np.linspace(0, 1, n_faixas + 1))
    cortes[0], cortes[-1] = -np.inf, np.inf
    return cortes

def indice_faixa(df, cortes):
    """Número da faixa de cada linha, a partir do preço esperado."""
    return pd.cut(df["previsto"], cortes, labels=False).astype(int)

def parametros_faixa(df_ref, cortes):
    """Centro e MAD do score em cada faixa de preço esperado."""
    score = calcular_score(df_ref)
    por_faixa = score.groupby(indice_faixa(df_ref, cortes))
    return {
        "centro": por_faixa.median(),
        "mad": por_faixa.apply(lambda s: (s - s.median()).abs().median()),
    }

def calcular_z_faixa(df, p, cortes):
    """z do score usando o centro e a MAD da faixa de preço esperado da linha."""
    faixa = indice_faixa(df, cortes)
    return (calcular_score(df) - faixa.map(p["centro"])) / (1.4826 * faixa.map(p["mad"]))

CONFIG_FAIXAS = {"só raiz (v2)": None, "5 faixas": 5, "10 faixas": 10, "20 faixas": 20}
KS_FAIXA = [2, 2.5, 3, 3.5, 4, 5]
LIMITE_FALSO_ALARME = 0.05
CENARIOS_ERRO = ["÷5", "÷3", "÷2", "x2", "x3", "x5", "x8"]

linhas = []
for nome, n_faixas in CONFIG_FAIXAS.items():
    cortes = cortes_faixas(df_oof["previsto"], n_faixas) if n_faixas else None
    z_cenario = {}
    for cenario, fator in FATORES.items():
        partes = []
        for f in folds_treino:
            ref = df_oof[df_oof["fold"] != f]
            alvo = injetar_erro(df_oof[df_oof["fold"] == f], fator)
            if n_faixas is None:
                z = calcular_z_cat(alvo, parametros_cat(ref, 10**9))   # só raiz
            else:
                z = calcular_z_faixa(alvo, parametros_faixa(ref, cortes), cortes)
            partes.append(z.abs())
        z_cenario[cenario] = pd.concat(partes)

    falso = {k: (z_cenario["x1 (original)"] > k).mean() * 100 for k in KS_FAIXA}
    aceitos = [k for k in KS_FAIXA if falso[k] <= LIMITE_FALSO_ALARME]
    linha = {"configuração": nome}
    if aceitos:
        k = min(aceitos)
        linha["k escolhido"] = k
        linha["falso alarme (%)"] = falso[k]
        for c in CENARIOS_ERRO:
            linha[f"recall {c} (%)"] = (z_cenario[c] > k).mean() * 100
        linha["recall médio (%)"] = np.mean([linha[f"recall {c} (%)"] for c in CENARIOS_ERRO])
    else:
        linha["k escolhido"] = np.nan
    linhas.append(linha)

tabela_faixas = pd.DataFrame(linhas).set_index("configuração")
print("=== Treino (parâmetros de outros folds): k escolhido e recall por configuração ===")
print("(critério: menor k com falso alarme <= 0,05%; erros: recall | x1: falso alarme)\n")
print(tabela_faixas.round(3).T.to_string())

### Evolução 8: centro e MAD por faixa de preço esperado (testada e descartada)

**Antes:** centro e MAD por categoria raiz, com k = 3. No treino, com parâmetros vindos de outros folds: falso alarme de 0,037% e recall médio de 35,3% nos erros sintéticos (58,3% em ÷5, 57,3% em ×5 e 81,6% em ×8).

**O que foi alterado:** dividimos as linhas do treino em 5, 10 ou 20 faixas pelo preço esperado (quantis do previsto fora do fold) e calculamos o centro e a MAD do score em cada faixa, em vez de por categoria raiz. Os parâmetros de cada fold vêm dos outros folds, e o k foi escolhido pelo mesmo critério de antes (falso alarme de no máximo 0,05%). O teste não foi usado.

**Diagnóstico prévio:** a MAD do score por faixa de preço esperado fica entre 0,33 e 0,355 em 8 das 10 faixas (0,282 na mais barata e 0,323 na mais cara), com centros entre −0,04 e +0,03. Por unidade, a MAD fica entre 0,335 e 0,345.

**Depois:**

| Configuração | k escolhido | Falso alarme | Recall ÷5 | Recall ×5 | Recall ×8 | Recall médio |
|---|---|---|---|---|---|---|
| Só raiz (v2) | 3 | 0,037% | 58,3% | 57,3% | 81,6% | 35,33% |
| 5 faixas | 3 | 0,025% | 58,3% | 57,1% | 81,6% | 35,23% |
| 10 faixas | 3 | 0,035% | 58,7% | 57,2% | 81,5% | 35,35% |
| 20 faixas | 3,5 | 0,004% | 41,0% | 38,6% | 69,8% | 24,32% |

**Ganho:** nenhum. A melhor versão (10 faixas) fica 0,02 ponto percentual acima da atual, dentro do ruído de medição.

**Decisão:** manter a MAD por categoria raiz. O ruído do preço é praticamente o mesmo em todos os níveis de preço e unidades, e por isso um limite mais fino não ajuda.

In [0]:
# ------------------------------------------------------------
# Evolução 9: hiperparâmetros do LightGBM (grade pequena)
# Testa número de folhas, mínimo de linhas por folha e número de árvores.
# Mede o MAE do treino fora do fold (4 folds, divisão por SKU), igual ao da MAD v2.
# Critério fixado antes: só vale mudar o modelo se o MAE melhorar pelo menos R$ 0,05.
# O teste NÃO é usado. Não altera nada no lake.
# ------------------------------------------------------------
GANHO_MINIMO_MAE = 0.05
GRADE = [
    {"num_leaves": nl, "min_child_samples": mcs, "n_estimators": ne}
    for nl in (15, 31, 63) for mcs in (100, 300) for ne in (68, 150)
]

resultados_hp = []
for params in GRADE:
    previsto_hp = pd.Series(np.nan, index=treino.index)
    for f in folds_treino:
        fora = treino["fold"] == f
        m = lgb.LGBMRegressor(
            objective="regression_l1", learning_rate=0.05,
            random_state=42, verbose=-1, **params,
        )
        m.fit(treino.loc[~fora, FEATURES_BASE], treino.loc[~fora, "preco"],
              categorical_feature=CATEGORICAS_BASE)
        previsto_hp[fora] = m.predict(treino.loc[fora, FEATURES_BASE])
    resultados_hp.append({**params, "MAE (R$)": mae(treino["preco"], previsto_hp)})

tabela_hp = pd.DataFrame(resultados_hp).sort_values("MAE (R$)").reset_index(drop=True)
atual = tabela_hp[(tabela_hp["num_leaves"] == 31) & (tabela_hp["min_child_samples"] == 100)
                  & (tabela_hp["n_estimators"] == N_ARVORES_BASE)]["MAE (R$)"].iloc[0]
tabela_hp["ganho vs atual (R$)"] = atual - tabela_hp["MAE (R$)"]

print("MAE da configuração atual (esperado 9,23): R$", round(atual, 3))
print("\n=== Grade de hiperparâmetros (MAE no treino fora do fold) ===")
print(tabela_hp.round(3).to_string(index=False))
print("\nMelhor ganho: R$", round(tabela_hp["ganho vs atual (R$)"].max(), 3),
      "| mínimo exigido: R$", GANHO_MINIMO_MAE)

### Evolução 9: hiperparâmetros do LightGBM (testada e descartada)

**Antes:** 31 folhas, mínimo de 100 linhas por folha e 68 árvores (learning rate 0,05). MAE no treino fora do fold (4 folds, divisão por SKU): R\$ 9,232.

**O que foi alterado:** grade de 12 configurações: número de folhas (15, 31 e 63), mínimo de linhas por folha (100 e 300) e número de árvores (68 e 150). Critério fixado antes dos números: só mudar o modelo se o MAE melhorar pelo menos R\$ 0,05. O teste não foi usado.

**Depois:**

| Configuração (folhas / mínimo por folha / árvores) | MAE no treino fora do fold |
|---|---|
| 15 / 100 / 150 (melhor) | R\$ 9,224 |
| 31 / 100 / 150 | R\$ 9,227 |
| 31 / 100 / 68 (atual) | R\$ 9,232 |
| 15 / 100 / 68 (pior) | R\$ 9,239 |

**Ganho:** R\$ 0,008 (cerca de 0,09%), abaixo do mínimo exigido de R\$ 0,05. Na prática, todas as configurações ficam entre R\$ 9,224 e R\$ 9,239.

**Decisão:** manter o modelo atual. O MAE quase não responde aos hiperparâmetros, o que combina com o diagnóstico do histórico do SKU: o ruído do preço, e não o ajuste do modelo, limita o resultado.

### Registro de evoluções do baseline

Cada linha mostra o resultado antes, o que foi alterado, o resultado depois e o ganho. As linhas 0 a 5 foram medidas no teste (765 SKUs que o modelo não viu), com erros sintéticos nas linhas normais. As linhas 6 a 9 são ideias testadas só no treino (parâmetros vindos de outros folds), e o teste não foi usado nelas, porque nenhuma superou a versão atual.

| # | O que foi alterado | Antes | Depois | Ganho | Notebook |
|---|---|---|---|---|---|
| 0 | Ponto de partida (baseline do notebook 09): LightGBM com 7 colunas, 68 árvores, k = 5, MAD calculada no treino | MAE R\$ 9,45; MAD global 0,336; 1.334 alertas; recall: 8,0% em ÷5, 0,4% em ×5 e 13,0% em ×8 | Referência para todas as linhas abaixo | Não se aplica | 09 |
| 1 | Colunas do modelo: de 12 (com marca e variante) para 7 (categoria, categoria pai, unidade e 4 medidas de tamanho) | MAE R\$ 9,53 | MAE R\$ 9,45 | −R\$ 0,08 (cerca de 0,8%) | 09 |
| 2 | Limite do alerta: k = 5 para k = 3 (escolhido no treino, com falso alarme de no máximo 0,05%) | Recall: 8,0% em ÷5, 0,4% em ×5 e 13,0% em ×8 | Recall: 57,6% em ÷5, 57,9% em ×5 e 81,6% em ×8 | +49,6, +57,5 e +68,6 pontos percentuais; rótulos reais seguem em 100%, com 9 alertas a mais em 58.546 linhas | 10 |
| 3 | MAD calculada com previsões fora do fold (em vez das linhas que o modelo já viu) | MAD global 0,336 | MAD global 0,338 | Sem ganho relevante; mantida por ser o cálculo mais defensável | 10 |
| 4 | Dois níveis de alerta: forte (\|z\| acima de 5) e revisão (\|z\| entre 3 e 5) | Um só nível | Forte: 1.334 linhas; revisão: 9 linhas | O recall total não muda (57,6% em ÷5); passa a separar o forte (7,7%) do que vai para revisão (49,9%) | 10 |
| 5 | Segunda técnica: Isolation Forest no lugar do LightGBM, com o mesmo limite de falso alarme no treino | Isolation Forest: 22,1% em ÷5, 6,7% em ×5 e 13,0% em ×8; 17 falsos alarmes | LightGBM: 57,6% em ÷5, 57,9% em ×5 e 81,6% em ×8; 9 falsos alarmes | LightGBM pega +35,5, +51,2 e +68,6 pontos percentuais a mais | 11 |
| 6 | MAD e centro por categoria (123 grupos, com mínimo de SKUs e volta à raiz abaixo disso) em vez de por categoria raiz | Só raiz: k = 3; recall médio 35,3% (58,3% em ÷5, 57,3% em ×5 e 81,6% em ×8) | Categoria com mín. 10 SKUs: k = 3,5; recall médio 24,3% (41,1%, 38,5% e 69,6%) | Nenhum: piorou cerca de 11 pontos percentuais; descartada | 10 |
| 7 | Limite por SKU (centro e MAD do próprio SKU, calculados na outra metade das observações) | Só raiz: k = 3; recall médio 35,3% | SKU misturado com a raiz: k = 5; recall médio 5,3%. SKU puro: nenhum k respeita o limite de falso alarme | Nenhum: piorou muito; descartada | 10 |
| 8 | Centro e MAD por faixa de preço esperado (5, 10 e 20 faixas) | Só raiz: recall médio 35,33% | 10 faixas: 35,35% (5 faixas: 35,23%; 20 faixas: 24,32%) | Nenhum: +0,02 ponto percentual, dentro do ruído; descartada | 10 |
| 9 | Hiperparâmetros do LightGBM (12 configurações de folhas, mínimo por folha e árvores) | MAE R\$ 9,232 no treino fora do fold | Melhor: 15 folhas, 100 por folha e 150 árvores, MAE R\$ 9,224 | −R\$ 0,008 (0,09%), abaixo do mínimo de R\$ 0,05; descartada | 10 |

**Diagnósticos (sem mudança no modelo):**

| Diagnóstico | Resultado |
|---|---|
| Comparar o preço com a mediana do próprio SKU | MAD 0,324 (contra 0,338 do modelo); recall de 64,9% em ÷5, 62,2% em ×5 e 84,8% em ×8, com falso alarme de 0,05% |
| Efeito da data da observação | MAD de 0,338 para 0,337; variação entre datas de 0,005 |
| Persistência do preço do mesmo SKU | MAD de 0,423 contra a observação anterior (perto dos 0,46 de observações independentes) |
| Erros reais rotulados por SKU | 6.551 linhas em 2.946 dos 3.805 SKUs; mediana de 2 por SKU (2,5% das observações): erros pontuais |

**Ponto de atenção:** uma regra simples (mediana por subcategoria e tamanho) fica no mesmo nível do LightGBM (62,4% em ÷5, 61,4% em ×5 e 84,1% em ×8, com 13 falsos alarmes). As evoluções 6 a 9 e os diagnósticos explicam o porquê: o preço de um mesmo SKU varia cerca de 32% entre observações, de forma parecida em todas as categorias, faixas de preço e datas, e mesmo conhecendo o histórico do próprio SKU o ganho é de só 3 a 6 pontos. Com as colunas atuais, o modelo está perto do limite dos dados; melhorar mais exige informação nova que explique a variação do preço.

### Conclusões da evolução do baseline

**O que foi feito:** partindo do baseline do notebook 09 (LightGBM, 7 colunas, 68 árvores, MAE de R\$ 9,45 nas linhas normais do teste), medimos o alerta em erros moderados injetados de forma sintética, corrigimos o cálculo da MAD com previsões fora do fold, escolhemos o limite k sem usar o teste, implementamos dois níveis de alerta e testamos outras quatro formas de melhorar o alerta (evoluções 6 a 9), nenhuma com ganho.

1. **Erros sintéticos:** multiplicamos o preço das 57.212 linhas normais do teste por fatores de ÷5 a ×8. Como todas as linhas passam a ser erros de tamanho conhecido, a % de linhas em alerta é o recall; no cenário original (×1) é o falso alarme.

2. **O baseline (k = 5) pega pouco erro moderado:** recall de 8,0% em ÷5, 0,2% em ÷3, 0,4% em ×5 e 13,0% em ×8, e 0% em ÷2, ×2 e ×3. Ele cobre os erros extremos dos rótulos (razão preço real / esperado menor ou igual a 0 ou maior ou igual a 14), mas deixa passar o que fica abaixo disso.

3. **MAD fora do treino:** treinamos o modelo base 4 vezes, cada uma deixando um fold de fora, e recalculamos a MAD com essas previsões. A MAD global foi de 0,336 para 0,338 (diferença de até 1,6% por categoria raiz) e o MAE no treino foi de R\$ 9,20 para R\$ 9,23. O otimismo da MAD do baseline era desprezível. Mantivemos a versão 2 por ser o cálculo mais defensável, mas ela não é a fonte do ganho.

4. **Escolha do k:** com erros sintéticos nas linhas do treino (previsões fora do fold), definimos o critério antes de ver os números: o menor k com falso alarme de no máximo 0,05% nas linhas normais. Resultado: k = 3 (0,032%); k = 2,5 deu 0,43% e k = 2 deu 4,8%.

5. **Resultado no teste (765 SKUs que o modelo não viu), baseline contra k = 3:** recall de 8,0% para 57,6% em ÷5; de 0,2% para 25,9% em ÷3; de 0% para 9,9% em ÷2; de 0% para 14,3% em ×3; de 0,4% para 57,9% em ×5; de 13,0% para 81,6% em ×8. Contra os rótulos reais: 1.343 alertas (9 a mais que no baseline), recall de 100% nos dois rótulos e precision de 99,3% contra o rótulo de apoio.

6. **Os 9 alertas extras:** todos são de preço abaixo do esperado (22% a 31% do esperado e da mediana do próprio SKU), com |z| entre 3,01 e 3,13 (no limite), e dois SKUs aparecem duas vezes com preços parecidos. São plausíveis, mas não são registros claramente anômalos. Só o negócio pode dizer se são promoção ou erro.

7. **Dois níveis de alerta (aprovados pelo Geovany e implementados na célula 13):** alerta forte (|z| acima de 5), com os mesmos 1.334 alertas do baseline, 100% dos dois rótulos e nenhum alerta sem rótulo; e alerta de revisão (|z| entre 3 e 5), com 9 linhas a mais em 58.546 (0,015%). Nos erros sintéticos, o forte pega 7,7% dos ÷5 e 12,0% dos ×8, e a revisão traz mais 49,9% e 69,6%: o ganho nos erros moderados está quase todo na revisão. O k = 3,5 continua como alternativa conservadora, com 0 alertas fora dos rótulos e recall de 70,3% em ×8 e 39,3% em ×5.

8. **Limitações:** erros de 2 vezes continuam sem ser detectados (0,2% em ×2), porque o ruído natural de cerca de um terço por observação os esconde. Os erros sintéticos não substituem rótulos reais de erros moderados. Os falsos alarmes são um teto, pois as linhas normais podem conter erros que nenhuma regra marcou. O MAE não mudou (R\$ 9,45), porque o modelo é o mesmo: a evolução está no alerta. As evoluções 6 a 9 foram medidas só no treino.

9. **Outras formas de melhorar o alerta (evoluções 6 a 9), todas sem ganho:** testamos MAD por categoria (123 grupos), limite por SKU, limite por faixa de preço esperado e hiperparâmetros do LightGBM. A MAD por categoria deu recall médio de 24,3% contra 35,3% da raiz; o limite por SKU deu 5,3%; o limite por faixa de preço deu 35,35%, igual ao atual; e a grade de hiperparâmetros melhorou o MAE em só R\$ 0,008, contra os R\$ 0,05 exigidos. Mantivemos o modelo e o limite atuais.

10. **O ruído natural do preço limita o alerta:** mesmo comparando cada preço com a mediana do próprio SKU, a MAD fica em 0,324 (contra 0,338 do modelo), e o recall sobe de 3 a 6 pontos percentuais. A data da observação não explica nada (variação entre datas de 0,005) e as observações do mesmo SKU são quase independentes (MAD de 0,423 contra a observação anterior, perto dos 0,46 esperados). Os erros reais rotulados são pontuais (mediana de 2 linhas por SKU, 2,5% das observações), então juntar observações do mesmo SKU não ajuda. Isso explica por que a regra simples empata com o LightGBM.

11. **Próximos passos:** levar os alertas de revisão ao negócio para saber se são erro ou promoção; buscar informação nova que explique por que o preço do mesmo SKU muda de uma observação para outra (as colunas atuais da tabela já foram testadas); e confirmar com o negócio se os erros moderados que importam são pontuais, como os rotulados.